In [2]:
import pandas as pd
import os
from tqdm import tqdm

In [3]:
label_maps = pd.read_csv("cvdf_filtered_labels_2.csv")
label_maps

,Tag,Category,Label
0,acrobatics,action,0
1,acting_in_play,action,1
2,adjusting_glasses,action,2
3,alligator_wrestling,action,3
4,american_football,action,4
...,...,...,...
2726,wetland,scene,2726
2727,wildlife_region,scene,2727
2728,workshop,scene,2728
2729,yard,scene,2729


In [4]:
categories = list(label_maps["Category"].unique())
categories

['action', 'attribute', 'concept', 'event', 'object', 'scene']

In [5]:
for category in categories:
    cat_map = label_maps[label_maps['Category'] == category].copy()
    cat_map["Label"] = range(len(cat_map))
    cat_map.to_csv(f"{category}_label_map.csv", index=False)

In [6]:
df = pd.read_csv("./hvu_cvdf_val_all_labels2.csv")
df = df.drop(columns=['contains_label', 'label'])

In [7]:
df

,Tags,youtube_id,time_start,time_end,original_filename
0,striking_combat_sports|uniform|competition|jap...,gD_G1b0wV5I,101.5,103.5,gD_G1b0wV5I_101.5_103.5.mp4
1,house|plant|sky|residential_area|land_lot|tree...,eeEC-oWWAM0,90.5,92.5,eeEC-oWWAM0_90.5_92.5.mp4
2,water|swimmer|floating|leisure|playing_water_p...,Np0jqmKPvSQ,170.5,172.5,Np0jqmKPvSQ_170.5_172.5.mp4
3,percussion|cymbal|drumhead|metal|skin_head_per...,s2I4-ErSc7E,182.0,184.0,s2I4-ErSc7E_182.0_184.0.mp4
4,fixing_the_roof|wood,TLZXKaS2abQ,31.0,33.0,TLZXKaS2abQ_31.0_33.0.mp4
...,...,...,...,...,...
11671,lighting|finger|moustache|facial_hair|conversa...,Ojk4Q7bS-VM,5.0,15.0,Ojk4Q7bS-VM_000005_000015.mp4
11672,massaging_neck|finger|neck|service|skin|girl|p...,vOFCraGyYPo,467.0,477.0,vOFCraGyYPo_000467_000477.mp4
11673,finger|packaging_and_labeling|cardboard_box|pa...,gGfJtwK6GZk,21.0,31.0,gGfJtwK6GZk_000021_000031.mp4
11674,dancer|choreography|dance|fun|girl|footwear|pe...,x5lYDVar2Tg,52.0,62.0,x5lYDVar2Tg_000052_000062.mp4


In [8]:
len(df['Tags'].str.split("|").explode().str.strip().unique())

2541

In [9]:
def get_categories(tags, tag_to_category):
    """Get the list of categories corresponding to the provided tags."""
    return [tag_to_category[tag] for tag in tags.split('|') if tag in tag_to_category]

In [10]:
def map_tags_to_categories(tags_file):
    """Create a dictionary mapping tags to their categories from the tags file."""
    tags_categories_df = pd.read_csv(tags_file)
    return dict(zip(tags_categories_df['Tag'], tags_categories_df['Category']))

In [11]:
def process_items_all_categories(tags, tag_to_category):
    """Process the tags to extract and join categories without filtering for main categories."""
    categories = get_categories(tags, tag_to_category)
    return '|'.join(categories)

In [12]:
tag_to_category = map_tags_to_categories("./cvdf_filtered_labels.csv")
df['Categories'] = df['Tags'].apply(lambda tags: process_items_all_categories(tags, tag_to_category))

In [13]:
df

,Tags,youtube_id,time_start,time_end,original_filename,Categories
0,striking_combat_sports|uniform|competition|jap...,gD_G1b0wV5I,101.5,103.5,gD_G1b0wV5I_101.5_103.5.mp4,concept|object|event|concept|action|concept|co...
1,house|plant|sky|residential_area|land_lot|tree...,eeEC-oWWAM0,90.5,92.5,eeEC-oWWAM0_90.5_92.5.mp4,scene|object|scene|scene|scene|object|object|s...
2,water|swimmer|floating|leisure|playing_water_p...,Np0jqmKPvSQ,170.5,172.5,Np0jqmKPvSQ_170.5_172.5.mp4,object|object|attribute|attribute|action|attri...
3,percussion|cymbal|drumhead|metal|skin_head_per...,s2I4-ErSc7E,182.0,184.0,s2I4-ErSc7E_182.0_184.0.mp4,action|object|object|attribute|object|object|o...
4,fixing_the_roof|wood,TLZXKaS2abQ,31.0,33.0,TLZXKaS2abQ_31.0_33.0.mp4,action|attribute
...,...,...,...,...,...,...
11671,lighting|finger|moustache|facial_hair|conversa...,Ojk4Q7bS-VM,5.0,15.0,Ojk4Q7bS-VM_000005_000015.mp4,concept|object|object|object|event|concept|act...
11672,massaging_neck|finger|neck|service|skin|girl|p...,vOFCraGyYPo,467.0,477.0,vOFCraGyYPo_000467_000477.mp4,action|object|object|concept|object|object|obj...
11673,finger|packaging_and_labeling|cardboard_box|pa...,gGfJtwK6GZk,21.0,31.0,gGfJtwK6GZk_000021_000031.mp4,object|concept|object|action|object|attribute|...
11674,dancer|choreography|dance|fun|girl|footwear|pe...,x5lYDVar2Tg,52.0,62.0,x5lYDVar2Tg_000052_000062.mp4,object|action|action|attribute|object|object|c...


In [14]:
def create_separate_dfs(dataset, main_categories, output_folder='./'):
    """Create separate CSV files for each main category in the dataset."""
    category_data = {category: [] for category in main_categories}
    
    for _, row in dataset.iterrows():
        tags = row['Tags'].split('|')
        categories = row['Categories'].split('|')
        youtube_id = row['youtube_id']
        time_start = row['time_start']
        time_end = row['time_end']
        original_filename = row['original_filename']
        
        for tag, category in zip(tags, categories):
            if category in main_categories:
                category_data[category].append([youtube_id, time_start, time_end, tag, original_filename])
    
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)
    
    for category, data in tqdm(category_data.items()):
        if data:  # Only create files for categories that have data
            category_df = pd.DataFrame(data, columns=['youtube_id', 'time_start', 'time_end', 'tag', 'original_filename'])
            output_file_path = os.path.join(output_folder, f'{category}.csv')
            category_df.to_csv(output_file_path, index=False)


In [15]:
create_separate_dfs(df, main_categories = categories, output_folder='./try2_val')

100%|█████████████████████████████████████████████| 6/6 [00:00<00:00, 10.22it/s]


In [57]:
action = pd.read_csv("./try1/action.csv")
action_label_map = pd.read_csv("./label_maps_cvdf/action_label_map.csv")

In [58]:
# action['label'] = action['tag'].map(action_label_map['Label'])

In [59]:
action_label_map

,Tag,Category,Label
0,acrobatics,action,0
1,acting_in_play,action,1
2,adjusting_glasses,action,2
3,alligator_wrestling,action,3
4,alpine_skiing,action,4
...,...,...,...
353,wicker_weaving,action,353
354,winking,action,354
355,wood_burning_art_,action,355
356,worship,action,356


In [60]:
# Merge the two dataframes on the 'tag' column from df1 and 'Tag' column from df2
merged_df = pd.merge(action, action_label_map, left_on="tag", right_on="Tag", how="left")

In [62]:
merged_df = merged_df.drop(columns=['Tag'])

In [63]:
merged_df['Label']

,youtube_id,time_start,time_end,tag,original_filename,Category,Label
0,gD_G1b0wV5I,101.5,103.5,doing_karate,gD_G1b0wV5I_101.5_103.5.mp4,action,88
1,eeEC-oWWAM0,90.5,92.5,powerbocking,eeEC-oWWAM0_90.5_92.5.mp4,action,229
2,eeEC-oWWAM0,90.5,92.5,recreation,eeEC-oWWAM0_90.5_92.5.mp4,action,247
3,Np0jqmKPvSQ,170.5,172.5,playing_water_polo,Np0jqmKPvSQ_170.5_172.5.mp4,action,222
4,Np0jqmKPvSQ,170.5,172.5,swimming,Np0jqmKPvSQ_170.5_172.5.mp4,action,304
...,...,...,...,...,...,...,...
22496,x5lYDVar2Tg,52.0,62.0,dance,x5lYDVar2Tg_000052_000062.mp4,action,80
22497,x5lYDVar2Tg,52.0,62.0,square_dancing,x5lYDVar2Tg_000052_000062.mp4,action,294
22498,x5lYDVar2Tg,52.0,62.0,recreation,x5lYDVar2Tg_000052_000062.mp4,action,247
22499,kDjlVB769V8,144.0,154.0,smile,kDjlVB769V8_000144_000154.mp4,action,284


In [73]:
merged_df['Label'] = merged_df['Label'].astype(str)
merged_df

,youtube_id,time_start,time_end,tag,original_filename,Category,Label
0,gD_G1b0wV5I,101.5,103.5,doing_karate,gD_G1b0wV5I_101.5_103.5.mp4,action,88
1,eeEC-oWWAM0,90.5,92.5,powerbocking,eeEC-oWWAM0_90.5_92.5.mp4,action,229
2,eeEC-oWWAM0,90.5,92.5,recreation,eeEC-oWWAM0_90.5_92.5.mp4,action,247
3,Np0jqmKPvSQ,170.5,172.5,playing_water_polo,Np0jqmKPvSQ_170.5_172.5.mp4,action,222
4,Np0jqmKPvSQ,170.5,172.5,swimming,Np0jqmKPvSQ_170.5_172.5.mp4,action,304
...,...,...,...,...,...,...,...
22496,x5lYDVar2Tg,52.0,62.0,dance,x5lYDVar2Tg_000052_000062.mp4,action,80
22497,x5lYDVar2Tg,52.0,62.0,square_dancing,x5lYDVar2Tg_000052_000062.mp4,action,294
22498,x5lYDVar2Tg,52.0,62.0,recreation,x5lYDVar2Tg_000052_000062.mp4,action,247
22499,kDjlVB769V8,144.0,154.0,smile,kDjlVB769V8_000144_000154.mp4,action,284


In [76]:
# Group by the 'Filename' column and aggregate the 'Label' and 'Label value' columns
grouped_df = merged_df.groupby('original_filename').agg({
    'Label': '|'.join,
    'tag': '|'.join
}).reset_index()

In [77]:
grouped_df

,original_filename,Label,tag
0,--RREBY3XLI_000013_000023.mp4,153,lifting_hat
1,--XWWLL8Spk_000000_000010.mp4,48,changing_gear_in_car
2,--uD0h1Rpvc_000024_000034.mp4,212,playing_marbles
3,-0QQoIK10U8_000020_000030.mp4,270,shaping_bread_dough
4,-0qOFqf_eRk_000016_000026.mp4,126,hand_washing_clothes
...,...,...,...
11789,zyoKhVFNEhg_000001_000011.mp4,80|61|170|124|55|193,dance|contorting|modern_dance|gymnastics|chore...
11790,zzgNs8euH2M_129.5_131.5.mp4,184,painting_fence
11791,zzgNs8euH2M_16.5_18.5.mp4,247|184,recreation|painting_fence
11792,zzgNs8euH2M_47.5_49.5.mp4,247|184,recreation|painting_fence


In [16]:
for category in categories:
    category_ds = pd.read_csv(f"./try2_val/{category}.csv")
    category_label_map = pd.read_csv(f"./label_maps_cvdf2/{category}_label_map.csv")
    merged_df = pd.merge(category_ds, category_label_map, left_on="tag", right_on="Tag", how="left")
    merged_df = merged_df.drop(columns=['Tag'])
    merged_df['Label'] = merged_df['Label'].astype(str)
    grouped_df = merged_df.groupby('original_filename').agg({
        'Label': '|'.join,
        'tag': '|'.join
    }).reset_index()
    print(len(set(list(category_ds['original_filename']))) == len(list(grouped_df['original_filename'])))
    grouped_df.to_csv(f"./video_label/val/{category}_zeroshot_ds.csv", index=False)

True
True
True
True
True
True


### Do it for training now

In [17]:
df = pd.read_csv("./hvu_cvdf_train_all_labels2.csv")
df = df.drop(columns=['contains_label', 'label'])

In [18]:
df.head()

,Tags,youtube_id,time_start,time_end,original_filename
0,child|loudspeaker|fun|speaker|electronic_devic...,3Qm_A6zhKSo,0.0,10.0,3Qm_A6zhKSo_000000_000010.mp4
1,concrete|photograph|child|fun|waving_hand|blue,7BbDqZc_tPU,0.0,10.0,7BbDqZc_tPU_000000_000010.mp4
2,performance|musical_instrument|performing_arts...,gfQUg9PHs14,300.0,310.0,gfQUg9PHs14_000300_000310.mp4
3,glasses|night|fun|musician|lighting|hair|human...,lx2-z9f-Jho,0.0,10.0,lx2-z9f-Jho_000000_000010.mp4
4,mouth|hair|smile|cheek|glasses|nose|head|eye|c...,D1666cuDZts,17.0,27.0,D1666cuDZts_000017_000027.mp4


In [19]:
len(df['Tags'].str.split("|").explode().str.strip().unique())

2723

In [20]:
tag_to_category = map_tags_to_categories("./cvdf_filtered_labels_2.csv")
df['Categories'] = df['Tags'].apply(lambda tags: process_items_all_categories(tags, tag_to_category))

In [21]:
df

,Tags,youtube_id,time_start,time_end,original_filename,Categories
0,child|loudspeaker|fun|speaker|electronic_devic...,3Qm_A6zhKSo,0.0,10.0,3Qm_A6zhKSo_000000_000010.mp4,object|object|attribute|object|object|action|o...
1,concrete|photograph|child|fun|waving_hand|blue,7BbDqZc_tPU,0.0,10.0,7BbDqZc_tPU_000000_000010.mp4,attribute|action|object|attribute|action|attri...
2,performance|musical_instrument|performing_arts...,gfQUg9PHs14,300.0,310.0,gfQUg9PHs14_000300_000310.mp4,concept|object|concept|concept|object|event|ob...
3,glasses|night|fun|musician|lighting|hair|human...,lx2-z9f-Jho,0.0,10.0,lx2-z9f-Jho_000000_000010.mp4,object|event|attribute|object|concept|object|o...
4,mouth|hair|smile|cheek|glasses|nose|head|eye|c...,D1666cuDZts,17.0,27.0,D1666cuDZts_000017_000027.mp4,object|object|action|object|object|object|obje...
...,...,...,...,...,...,...
201667,summer|sport|fun|sport_venue|games|competition...,0Ymc8xw8l_g,9.5,11.5,0Ymc8xw8l_g_9.5_11.5.mp4,event|concept|attribute|scene|concept|event|at...
201668,riding_bumper_cars|mode_of_transport|car|darkn...,iFWlWbpTm1g,34.5,36.5,iFWlWbpTm1g_34.5_36.5.mp4,action|concept|object|attribute|object|action|...
201669,downtown|neighbourhood|pedestrian|road|window|...,7CfgZITnsxs,17.0,19.0,7CfgZITnsxs_17.0_19.0.mp4,scene|scene|object|scene|object|scene|scene|ac...
201670,riding_bumper_cars|mode_of_transport|vehicle|a...,3B4AlI56wGA,11.5,13.5,3B4AlI56wGA_11.5_13.5.mp4,action|concept|object|scene|attribute|action|o...


In [22]:
create_separate_dfs(df, main_categories = categories, output_folder='./try2_train')

100%|█████████████████████████████████████████████| 6/6 [00:08<00:00,  1.43s/it]


In [23]:
for category in categories:
    category_ds = pd.read_csv(f"./try2_train/{category}.csv")
    category_label_map = pd.read_csv(f"./label_maps_cvdf2/{category}_label_map.csv")
    merged_df = pd.merge(category_ds, category_label_map, left_on="tag", right_on="Tag", how="left")
    merged_df = merged_df.drop(columns=['Tag'])
    merged_df['Label'] = merged_df['Label'].astype(str)
    grouped_df = merged_df.groupby('original_filename').agg({
        'Label': '|'.join,
        'tag': '|'.join
    }).reset_index()
    print(len(set(list(category_ds['original_filename']))) == len(list(grouped_df['original_filename'])))
    grouped_df.to_csv(f"./video_label/train/{category}_zeroshot_ds.csv", index=False)

True
True
True
True
True
True
